In [2]:
import os, re, cv2, torch, pandas as pd
import torchvision.models as models
import torchvision.transforms as transforms
from pathlib import Path


image_folder = Path(r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\AND_result_fold2")


def get_id(filename: str) -> int:
    return int(re.findall(r'\d+', Path(filename).stem)[0])


image_files = sorted(
    [f for f in os.listdir(image_folder) if f.lower().endswith(('.jpg', '.png'))],
    key=get_id
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

models_config = [
    ('resnet18',         models.resnet18,          'features_Resnet18.xlsx'),
    ('resnet50',         models.resnet50,          'features_Resnet50.xlsx'),
    ('efficientnet_b0',  models.efficientnet_b0,   'features_EfficientNetB0.xlsx'),
    ('mobilenet_v3_small', models.mobilenet_v3_small, 'features_MobileNetV3Small.xlsx'),
    ('vgg16',            models.vgg16,             'features_VGG16.xlsx'),
    ('densenet121',      models.densenet121,       'features_DenseNet121.xlsx'),
    ('inception_v3',     models.inception_v3,      'features_InceptionV3.xlsx'),
    ('vit_tiny', lambda: torch.hub.load('facebookresearch/dino:main', 'dino_vits16'), 'features_ViTTiny.xlsx')
]

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

for model_name, model_fn, output_name in models_config:

    if 'vit' in model_name:
        model = model_fn().to(device).eval()
    else:
        model = model_fn(pretrained=True).to(device).eval()
    
    all_rows = []          
    
    for img_file in image_files:
        img_path = image_folder / img_file
        img_id   = get_id(img_file)     
        
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_tensor = transform(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            feats = model(img_tensor).cpu().numpy().flatten()
        

        row = [img_id] + feats.tolist()
        all_rows.append(row)
    

    df = pd.DataFrame(all_rows)
    df.to_excel(output_name, index=False, header=False)
    print(f"{output_name} saved ✅")
